# Chapter 9 · Part 4 — Reaction representations and data checks

A reaction record tells us which molecular graphs are connected by a proposed transformation. Parts 1–3 asked about thermodynamics, paths, and rates. Here we ask a different, practical question: **does the stored reaction say what we think it says?**

By the end, you should be able to:

- distinguish reaction SMILES, atom mapping, and an RDKit reaction SMARTS template;
- check a complete net reaction for atom, isotope, and charge balance;
- identify changed bonds and charges without relying on atom order;
- enumerate and sanitize small product sets, remove duplicate matches, and preserve stereochemical uncertainty;
- explain why a graph transformation alone cannot establish a mechanism, rate, or yield.

**Runtime and inputs.** This standalone notebook uses embedded structures and a few RDKit graph operations. No downloads, quantum calculations, or reaction-database access are needed. A deliberately small product cap bounds enumeration. Generated data go to `outputs/chapter09_part4/`.

### Start here: three representations, three kinds of claim

| Representation | What it expresses | What a beginner should check |
|---|---|---|
| Reaction SMILES | Specific reactant, agent, and product molecular graphs | Were the physical species and byproducts recorded? |
| Atom mapping | A proposed correspondence between atoms before and after | Is every required label unique and chemically consistent? |
| Reaction SMARTS | A rule that matches graph patterns and constructs product graphs | Does a match merely generate a candidate, or is there evidence it occurs? |

A map label is like a luggage tag carried by one atom across the record; changing a bond does not change that atom's element. By contrast, a **mechanism** describes elementary molecular events. Solvent, temperature, pathways, and measured outcomes are additional information, not hidden inside a graph match.

**First pass:** read the reaction drawing and the audit matrix, then examine duplicate enumeration and stereochemistry. The small helper functions make the checks reusable; they are not a comprehensive reaction-validation system.

In [ ]:
from collections import Counter
from pathlib import Path
import json

import pandas as pd
from IPython.display import display
from rdkit import Chem, rdBase
from rdkit.Chem import Draw, rdChemReactions

OUT = Path("outputs/chapter09_part4")
OUT.mkdir(parents=True, exist_ok=True)
print("RDKit:", rdBase.rdkitVersion)

## 1. A complete net ionic reaction

Use chloride displacement of bromide from bromomethane as a small **representation example**:

$$
\mathrm{Cl^- + CH_3Br \longrightarrow CH_3Cl + Br^-}.
$$

Reaction SMILES has three fields: `reactants>agents>products`. A dot separates components, and `>>` means the agents field is empty. We include the bromide byproduct; omitted counterions are spectators on both sides of this net ionic equation. Calling a species an *agent* in a file does not prove that it is chemically unconsumed.

The numbers after colons are **atom-map labels**: label 2 identifies the same carbon on both sides. They are neither atomic numbers nor RDKit's zero-based atom indices. Mapping is supplied information, which can be wrong; it does not demonstrate an SN2 mechanism.

Source: [Daylight reaction-SMILES syntax](https://www.daylight.com/dayhtml/doc/theory/theory.smiles.html).

In [ ]:
mapped_record = "[Cl-:1].[CH3:2][Br:3]>>[Cl:1][CH3:2].[Br-:3]"
reaction_record = rdChemReactions.ReactionFromSmarts(mapped_record, useSmiles=True)
assert reaction_record is not None
warnings, errors = reaction_record.Validate()
assert (warnings, errors) == (0, 0)
display(Draw.ReactionToImage(reaction_record, subImgSize=(240, 180)))
print("Stored record:", mapped_record)

## 2. Audit a record before calculating a reaction energy

Our teaching auditor expects a **complete net equation with one copy of each listed component and every heavy atom mapped once on each side**. Implicit hydrogens are counted by adding explicit H atoms to a copy. Atom-map correspondence must conserve element and isotope; total charge must balance, although the charge of an individual mapped atom may change.

This scope is intentional: many reaction datasets store only the desired product, omit salts or byproducts, or use partial mapping. Failing this audit means a record is unsuitable *as a complete balanced equation*; it does not by itself prove that the reported laboratory transformation is false. Fractional stoichiometric coefficients, electrons supplied by an electrode, and explicit hydrogen mapping would need an expanded representation and audit.

RDKit sanitization checks molecular consistency. It does not establish chemical feasibility. Sources: [RDKit molecule operations](https://www.rdkit.org/docs/source/rdkit.Chem.rdmolops.html) and [reaction API](https://www.rdkit.org/docs/source/rdkit.Chem.rdChemReactions.html).

In [ ]:
def parse_side(text):
    molecules = [Chem.MolFromSmiles(part) for part in text.split(".")] if text else []
    if any(mol is None for mol in molecules):
        raise ValueError("A component could not be parsed and sanitized.")
    return molecules


def inventory(molecules):
    '''Count all atoms, including H; isotope 0 means unspecified isotope.'''
    atoms = Counter()
    charge = 0
    for mol in molecules:
        atoms.update((atom.GetSymbol(), atom.GetIsotope())
                     for atom in Chem.AddHs(mol).GetAtoms())
        charge += Chem.GetFormalCharge(mol)
    return atoms, charge


def mapped_heavy_atoms(molecules):
    result = {}
    for mol in molecules:
        for atom in mol.GetAtoms():
            if atom.GetAtomicNum() == 1:
                continue
            label = atom.GetAtomMapNum()
            if label <= 0 or label in result:
                raise ValueError("Heavy-atom maps must be positive and unique on each side.")
            result[label] = atom
    return result


def audit_record(record):
    fields = record.split(">")
    if len(fields) != 3:
        raise ValueError("Expected reactants>agents>products.")
    reactants, agents, products = [parse_side(field) for field in fields]
    if not reactants or not products:
        raise ValueError("Both reacting sides must be present.")
    left, right = mapped_heavy_atoms(reactants), mapped_heavy_atoms(products)
    if left.keys() != right.keys():
        raise ValueError("The heavy-atom map sets differ between sides.")
    for label in left:
        a, b = left[label], right[label]
        if (a.GetAtomicNum(), a.GetIsotope()) != (b.GetAtomicNum(), b.GetIsotope()):
            raise ValueError(f"Map {label} changes element or isotope.")
    left_atoms, left_charge = inventory(reactants)
    right_atoms, right_charge = inventory(products)
    if left_atoms != right_atoms:
        raise ValueError("The atom/isotope inventories differ, including hydrogen.")
    if left_charge != right_charge:
        raise ValueError("The total formal charge is not balanced.")
    return {"mapped_heavy_atoms": len(left), "net_charge": left_charge,
            "atom_count_including_H": sum(left_atoms.values()),
            "agent_components_excluded_from_balance": len(agents), "balanced": True}


audit = audit_record(mapped_record)
assert audit["net_charge"] == -1 and audit["atom_count_including_H"] == 6
display(pd.Series(audit, name="Complete net-reaction audit"))

In [ ]:
# Each example is syntactically readable but unsuitable as our complete net equation.
incomplete_records = {
    "omitted bromide": "[Cl-:1].[CH3:2][Br:3]>>[Cl:1][CH3:2]",
    "duplicate map": "[Cl-:1].[CH3:2][Br:2]>>[Cl:1][CH3:2].[Br-:3]",
    "wrong net charge": "[Cl-:1].[CH3:2][Br:3]>>[Cl:1][CH3:2].[Br:3]",
    "changed isotope": "[Cl-:1].[13CH3:2][Br:3]>>[Cl:1][CH3:2].[Br-:3]",
}
diagnostics = []
for label, record in incomplete_records.items():
    try:
        audit_record(record)
    except ValueError as exc:
        diagnostics.append({"problem": label, "diagnostic": str(exc)})
    else:
        raise AssertionError(f"The auditor unexpectedly accepted: {label}")
display(pd.DataFrame(diagnostics))

### Worked dataset decision: which checks fail together?

Imagine preparing reaction records for thermochemical calculations. A parser accepting the text is only the first step. Compare the complete record with the deliberately damaged examples using **separate, interpretable checks**, rather than one opaque quality score.

The matrix below checks unique positive heavy-atom maps, matching map sets, mapped element/isotope identity, full atom/isotope inventory (including H), and total charge. Gray means an identity comparison cannot be made because mapping prerequisites failed. Several checks may fail for the same record; the original auditor stops at its first error, whereas this visual exposes those overlaps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

audit_examples = {"Complete net record": mapped_record, **incomplete_records}
matrix_rows = []
for label, record in audit_examples.items():
    reactants, _, products = [parse_side(field) for field in record.split(">")]
    heavy_sides = [[a for mol in side for a in mol.GetAtoms() if a.GetAtomicNum() != 1]
                   for side in (reactants, products)]
    map_lists = [[a.GetAtomMapNum() for a in atoms] for atoms in heavy_sides]
    unique_maps = all(all(value > 0 for value in maps) and len(maps) == len(set(maps)) for maps in map_lists)
    same_maps = set(map_lists[0]) == set(map_lists[1])
    identity = None
    if unique_maps and same_maps:
        identities = [{a.GetAtomMapNum(): (a.GetAtomicNum(), a.GetIsotope()) for a in atoms} for atoms in heavy_sides]
        identity = identities[0] == identities[1]
    left_inventory, left_charge = inventory(reactants)
    right_inventory, right_charge = inventory(products)
    matrix_rows.append([unique_maps, same_maps, identity,
                        left_inventory == right_inventory, left_charge == right_charge])
audit_matrix = np.array([[0.5 if value is None else float(value) for value in row] for row in matrix_rows])
assert np.all(audit_matrix[0] == 1) and np.all(np.any(audit_matrix[1:] == 0, axis=1))
fig, ax = plt.subplots(figsize=(9, 4.7), layout="constrained")
cmap = ListedColormap(["#d99b8c", "#dedede", "#8dbfaf"])
ax.imshow(audit_matrix, cmap=cmap, norm=BoundaryNorm([-0.25, 0.25, 0.75, 1.25], 3), aspect="auto")
for i in range(len(matrix_rows)):
    for j in range(5):
        ax.text(j, i, {0.0: "Fail", 0.5: "N/A", 1.0: "Pass"}[audit_matrix[i, j]], ha="center", va="center", fontsize=9)
ax.set(xticks=range(5), xticklabels=["Unique positive\nmaps", "Same\nmap set", "Mapped element\n& isotope", "All-atom\ninventory", "Net\ncharge"],
       yticks=range(len(audit_examples)), yticklabels=list(audit_examples), title="A readable record can still fail chemical bookkeeping")
ax.tick_params(length=0)
ax.legend(handles=[Patch(color="#dedede", label="N/A: mapping prerequisites failed")],
          loc="upper center", bbox_to_anchor=(0.5, -0.15), frameon=False, fontsize=9)
fig.savefig(OUT / "reaction_audit_matrix.png", dpi=140)
plt.show()

**Conclusion.** Omitting a byproduct affects more than the displayed product list: it can break atom and charge balance and the mapping correspondence. Changing an isotope can preserve elemental counts while invalidating isotope-resolved bookkeeping. The matrix tells a curator which fields need investigation without inventing missing chemistry.

**Next step:** retain the raw record and source, resolve omissions when evidence permits, and flag records unsuitable for a complete reaction-energy calculation. Passing all five checks still cannot establish mechanism, experimental success, or yield. A balanced hypothetical reaction is still hypothetical.

## 3. Locate changes by map labels

Compare bonds with the same pair of mapped endpoints on each side. Missing bonds are broken or formed; a changed bond type is a bond-order change. Separately compare atomic formal charges. This is a **net graph edit**, which may summarize several elementary steps.

The simple comparison below is appropriate for these consistently represented, non-aromatic records. A general reaction pipeline also needs a consistent aromaticity and resonance policy before comparing bond types.

In [ ]:
def mapped_bonds(molecules):
    bonds = {}
    for mol in molecules:
        for bond in mol.GetBonds():
            a, b = bond.GetBeginAtom(), bond.GetEndAtom()
            if a.GetAtomicNum() == 1 or b.GetAtomicNum() == 1:
                continue
            key = tuple(sorted((a.GetAtomMapNum(), b.GetAtomMapNum())))
            bonds[key] = str(bond.GetBondType())
    return bonds


def reaction_changes(record):
    audit_record(record)
    reactants, _, products = [parse_side(field) for field in record.split(">")]
    before, after = mapped_bonds(reactants), mapped_bonds(products)
    changes = [
        {"kind": "bond", "maps": str(pair),
         "before": before.get(pair, "absent"), "after": after.get(pair, "absent")}
        for pair in sorted(before.keys() | after.keys())
        if before.get(pair) != after.get(pair)
    ]
    left, right = mapped_heavy_atoms(reactants), mapped_heavy_atoms(products)
    changes += [
        {"kind": "formal charge", "maps": str(label),
         "before": left[label].GetFormalCharge(), "after": right[label].GetFormalCharge()}
        for label in sorted(left)
        if left[label].GetFormalCharge() != right[label].GetFormalCharge()
    ]
    return changes


changes = reaction_changes(mapped_record)
assert mapped_bonds(parse_side(mapped_record.split(">>")[0])) == {(2, 3): "SINGLE"}
assert sum(change["kind"] == "bond" for change in changes) == 2
display(pd.DataFrame(changes))

## 4. A transformation template generates candidate graphs

An RDKit **reaction SMARTS** template uses substructure queries for its reactants. Our query asks for a neutral, four-connected carbon bonded to bromine and a chloride anion. Product atom-map labels specify correspondence to matched input atoms. RDKit reaction SMARTS, reaction SMILES, and Daylight SMIRKS are related but distinct languages.

We explicitly request neutral product chlorine, `[Cl+0:3]`. With RDKit's reaction rules, leaving a product property unspecified can preserve it from the matched reactant. Copying chloride's negative charge onto bonded chlorine would be incorrect here.

Sources: [RDKit Book: reaction SMARTS](https://www.rdkit.org/docs/RDKit_Book.html#reaction-smarts), [RDKit author's explanation of property changes](https://greglandrum.github.io/rdkit-blog/posts/2025-04-26-specifying-changes-in-reactions.html), and [Daylight SMIRKS](https://www.daylight.com/dayhtml/doc/theory/theory.smirks.html).

In [ ]:
substitution_smarts = "[C;+0;X4:1]-[Br:2].[Cl-:3]>>[C+0:1]-[Cl+0:3].[Br-:2]"
substitution = rdChemReactions.ReactionFromSmarts(substitution_smarts)
assert substitution.Validate() == (0, 0)
substitution.Initialize()
chloride = Chem.MolFromSmiles("[Cl-]")


def product_key(products):
    '''A sorted multiset of isomeric SMILES, ignoring arbitrary atom-map labels.'''
    smiles = []
    for product in products:
        copy = Chem.Mol(product)
        for atom in copy.GetAtoms():
            atom.SetAtomMapNum(0)
        smiles.append(Chem.MolToSmiles(copy, canonical=True, isomericSmiles=True))
    return tuple(sorted(smiles))


def enumerate_products(template, reactants, cap=20):
    outcomes = template.RunReactants(tuple(reactants), maxProducts=cap)
    if len(outcomes) >= cap:
        raise RuntimeError("Product cap reached; completeness is uncertain. Narrow the query.")
    unique = {}
    for products in outcomes:
        for product in products:
            Chem.SanitizeMol(product)
            Chem.AssignStereochemistry(product, cleanIt=True, force=True)
        if inventory(reactants) != inventory(products):
            raise ValueError("Generated products do not conserve atoms/isotopes and charge.")
        unique.setdefault(product_key(products), products)
    return outcomes, unique

`RunReactants` returns a tuple of product tuples: one bundle for each match. Each bundle here contains **both** the organic product and bromide. Products need sanitization before downstream molecular operations.

Multiple matches can generate the same bundle. Our identity key removes map labels, keeps stereochemistry, and sorts the components. It does not merge tautomers or different protonation states. The small cap prevents an unexpectedly broad query from creating a large result set. Source: [RDKit reaction API](https://www.rdkit.org/docs/source/rdkit.Chem.rdChemReactions.html).

In [ ]:
substrates = {
    "bromomethane": "CBr",
    "bromoethane": "CCBr",
    "tert-butyl bromide": "CC(C)(C)Br",
    "1,2-dibromoethane": "BrCCBr",
}
enumeration_rows = []
for name, smiles in substrates.items():
    molecule = Chem.MolFromSmiles(smiles)
    raw, unique = enumerate_products(substitution, [molecule, chloride])
    enumeration_rows.append({
        "substrate": name, "input_SMILES": smiles, "matched_outcomes": len(raw),
        "unique_bundles": len(unique),
        "product_bundles": [" . ".join(key) for key in unique],
    })
enumeration_table = pd.DataFrame(enumeration_rows)
assert enumeration_table["matched_outcomes"].tolist() == [1, 1, 1, 2]
assert enumeration_table["unique_bundles"].tolist() == [1, 1, 1, 1]
display(enumeration_table)
display(Draw.MolsToGridImage(
    [Chem.MolFromSmiles(s) for s in substrates.values()],
    legends=list(substrates), molsPerRow=4, subImgSize=(220, 170),
))

**Read the counterexamples.** The two ends of 1,2-dibromoethane give two equivalent matches and one unique monosubstitution bundle. Neither match count predicts product yield, nor does one application simulate reaction to completion.

The template also accepts tert-butyl bromide. That match does not support a concerted SN2 assignment: this carbon is sterically hindered, and mechanism and competition with elimination depend on substrate and conditions. The query intentionally lacks these physical constraints. Energetics from Part 1, path evidence from Part 2, and kinetic modeling from Part 3 answer questions a pattern match cannot.

No yields, barriers, or observed reaction outcomes are attached to this enumeration.

## 5. Encode inversion while preserving what is unknown

For a specified 2-bromobutane stereocenter, we can explicitly encode inversion in a template. In SMILES/SMARTS, `@` and `@@` describe a local neighbor ordering; they are not synonyms for R and S. Here the corresponding neighbor order is preserved between the two sides of the template, so changing `@` to `@@` requests inversion.

We check both enantiomers and an input with unspecified stereochemistry. The last input must remain unspecified. A missing stereo label is not evidence for a pure enantiomer or for a 50:50 racemate. Sources: [RDKit reaction stereochemistry](https://www.rdkit.org/docs/RDKit_Book.html#chirality) and [Daylight chirality convention](https://www.daylight.com/dayhtml/doc/theory/theory.smiles.html).

In [ ]:
inversion_smarts = (
    "[C:4][C@H:1]([Br:2])[C:5].[Cl-:3]"
    ">>[C:4][C@@H:1]([Cl+0:3])[C:5].[Br-:2]"
)
inversion = rdChemReactions.ReactionFromSmarts(inversion_smarts)
assert inversion.Validate() == (0, 0)
inversion.Initialize()


def center_labels(mol):
    return [label for _, label in Chem.FindMolChiralCenters(mol, includeUnassigned=True)]


stereo_rows, stereo_images, stereo_legends = [], [], []
for smiles in ["C[C@H](Br)CC", "C[C@@H](Br)CC", "CC(Br)CC"]:
    reactant = Chem.MolFromSmiles(smiles)
    raw, unique = enumerate_products(inversion, [reactant, chloride])
    assert len(unique) == 1
    products = next(iter(unique.values()))
    organic = next(mol for mol in products if any(a.GetAtomicNum() == 6 for a in mol.GetAtoms()))
    before, after = center_labels(reactant), center_labels(organic)
    expected = {"S": "R", "R": "S", "?": "?"}[before[0]]
    assert after == [expected]
    stereo_rows.append({"input": smiles, "input_CIP": before[0],
                        "organic_product": product_key([organic])[0], "product_CIP": after[0]})
    stereo_images.extend([reactant, organic])
    stereo_legends.extend([f"Input: {before[0]}", f"Template product: {after[0]}"])
display(pd.DataFrame(stereo_rows))
display(Draw.MolsToGridImage(stereo_images, legends=stereo_legends,
                           molsPerRow=2, subImgSize=(300, 180)))

For these particular structures, inversion changes S to R and R to S because the priority relationships remain comparable. Do not use a change in R/S label as a universal inversion detector: changing substituents can change CIP priority rankings. Nor should you compare isolated `@` characters from canonical strings, because their neighbor order can change during serialization.

This template imposes a stereochemical relationship. It does not calculate the path or establish that the transformation occurs under any specified conditions.

## 6. Keep raw records and transformation provenance

Save the input separately from generated identity keys, and record the toolkit version and exact template. Canonical SMILES is a useful key within a specified workflow; it is not a universal identifier independent of toolkit, version, aromaticity, tautomer, or protonation choices. Unspecified stereochemistry stays unspecified.

These outputs are small teaching records. If building a reaction dataset, also record experimental conditions, source attribution, measured versus inferred fields, and missing components. A dataset for learning reaction outcomes must not silently label every generated product as an experimental success.

In [ ]:
provenance = {
    "example_type": "teaching graph transformations; no experimental outcomes or yields",
    "rdkit_version": rdBase.rdkitVersion,
    "raw_mapped_reaction_SMILES": mapped_record,
    "complete_net_equation_audit": audit,
    "net_graph_changes": changes,
    "substitution_reaction_SMARTS": substitution_smarts,
    "inversion_reaction_SMARTS": inversion_smarts,
    "product_identity_policy": "sorted isomeric SMILES bundle; atom maps removed; no tautomer/protonation merging",
    "enumeration": enumeration_rows,
    "stereochemistry": stereo_rows,
}
record_path = OUT / "reaction_examples.json"
record_path.write_text(json.dumps(provenance, indent=2) + "\n", encoding="utf-8")
enumeration_table.to_csv(OUT / "enumeration.csv", index=False)
assert json.loads(record_path.read_text(encoding="utf-8")) == provenance
print("Saved:", record_path)

## Exercises

1. Change all map labels consistently from 1, 2, 3 to 10, 20, 30. Should the chemistry or the balance change? What if you change a label only on one side?
2. Why must bromide remain in the net equation when using it to compute a reaction free energy? Why is an omitted spectator counterion different?
3. The template accepts tert-butyl bromide. Which additional information would you need to judge mechanism, rate, and selectivity?
4. Why does 1,2-dibromoethane produce two matches but one unique product bundle? Can you infer a yield from either number?
5. Explain why `@`/`@@`, R/S, and geometric inversion are related but different descriptions. What does the unspecified input tell you?
6. A patent record contains only the principal product and no byproducts. How would you store that limitation without inventing a complete reaction equation?

<details>
<summary>Suggested answers</summary>

1. Consistent renumbering changes labels only. One-sided renumbering breaks the claimed atom correspondence, so this auditor rejects it.
2. Bromide contains atoms and charge that leave the organic substrate, and contributes to the chosen complete reaction's chemical potential. A spectator appearing in the same state and amount on both sides cancels from that equation; an omitted product does not.
3. Relevant evidence includes substrate geometry, nucleophile and leaving group, solvent, temperature, competing pathways, and appropriate measured rates or calculated free-energy barriers. A matching graph pattern supplies none of these by itself.
4. The two equivalent brominated ends lead to the same molecular bundle. Matches count graph embeddings, and unique bundles count represented outcomes; neither is an experimental amount.
5. `@`/`@@` depend on local atom order; R/S use CIP priorities; inversion concerns the spatial correspondence of substituents. An unspecified label means the record does not resolve stereochemistry at that center.
6. Preserve the original record and source, flag its incomplete product/stoichiometry fields, and avoid using it directly as a balanced thermochemical equation. Any later reconstruction needs separate provenance and uncertainty.

</details>

## Chapter 9 recap

| Question | Evidence developed in this chapter |
| --- | --- |
| What is the reaction's equilibrium tendency in specified states? | Balanced thermochemistry, activities, and standard states in [Part 1](Chapter09_Part1.ipynb) |
| Does a candidate saddle connect the proposed basins on this model surface? | Gradient, vibrational mode, short IRC segments, and endpoint refinement in [Part 2](Chapter09_Part2.ipynb) |
| What rates and product fractions follow from specified assumptions? | Consistent rate constants and kinetic equations in [Part 3](Chapter09_Part3.ipynb) |
| What exactly is represented or generated in a reaction record? | Mapping, balance, graph edits, sanitization, and provenance in this notebook |

[Previous: Part 3](Chapter09_Part3.ipynb) · [Course contents and setup](Readme.md) · [Chapter 9 revision notes](docs/chapter9-review.md)